# House Prices: repeated-CV feature/blend rank validation

## tl;dr

- 이미 제출된 네 변경 중 세 feature blend의 개선 방향은 CV·반복 OOF·3-seed ensemble OOF에서 모두 재현됐다.
- QualityLivingArea blend는 local과 public에서 모두 1위였다.
- 전체 5개 후보의 local/public 순위 상관은 Spearman 0.80, Kendall 0.60이며 10개 쌍 중 8개의 순서가 일치했다.
- 두 불일치는 TotalSF/TotalBathrooms의 중간 순서와 baseline/raw-year의 미세한 순서다.
- hardcode 제거는 local CV에서 0.000246 개선됐지만 public에서는 0.00026 악화되어 네 변화 중 유일하게 방향이 맞지 않았다.
- 따라서 반복 validation은 큰 feature/blend 방향에는 유용하지만 작은 전처리 차이나 근소한 후보 순위를 정확히 예측하지는 못한다.

## Context & Methods

이미 Kaggle에 제출한 네 변경의 public 움직임이, 앞서 선택한 반복 validation에서도
같은 방향과 순서로 나타나는지 확인한다. public 점수는 모델 학습과 blend weight
추정이 끝난 뒤 정렬 진단에만 사용한다.

### Key Assumptions

- 기준점: 역사적 BaseMLP (`Id=524, YearRemodAdd=2007` 모델링 복사본).
- 실험 1: `RAWYEAR-ONLY`, 즉 해당 보정을 제거하고 원본 2008을 유지.
- 실험 2~4: raw-year 모델과 `TotalSF`, `TotalBathrooms`,
  `QualityLivingArea` 모델의 log-space convex blend.
- validation: `KFold(5, shuffle=True)`를 split seed 42, 2026, 3407에서 반복.
- blend weight: 각 seed의 현재 validation fold를 제외한 나머지 OOF 행에서만 추정.
- public 비교: baseline 0.12579, raw 0.12605, TotalSF blend 0.12564,
  TotalBathrooms blend 0.12538, QualityLivingArea blend 0.12496.
- private leaderboard 점수는 모두 미확인이다.
- 외부 프로젝트 스크립트는 사용하지 않는다.

## Data

`data/train.csv` 1,460행을 사용한다. raw CSV는 수정하지 않고, 행 삭제 없이
fold마다 전처리기와 BaseMLP를 새로 적합한다. 기존 반복-CV baseline OOF는
동일한 validation anchor로 재사용하며 해시와 source 재현 검사를 수행한다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "validation_test" /
    "basemlp_feature_blend_rank_validation.ipynb"
)
REPORT_DIR = ROOT / "reports" / "validation_test"
ARTIFACT_DIR = ROOT / "artifacts" / "validation_test"
RUN_ID = "VALTEST-20260720-BASEMLP-FEATURE-RANK-NB-02"
MODEL_FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_feature_rank_model_fold_results.csv"
CANDIDATE_FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_feature_rank_candidate_fold_results.csv"
OOF_RESULTS_PATH = REPORT_DIR / "basemlp_feature_rank_oof_predictions.csv"
RESULTS_PATH = REPORT_DIR / "basemlp_feature_rank_results.csv"
HYPOTHESIS_RESULTS_PATH = REPORT_DIR / "basemlp_feature_rank_hypotheses.csv"
SUMMARY_PATH = REPORT_DIR / "basemlp_feature_rank_summary.json"
BASELINE_REPEATED_OOF_PATH = REPORT_DIR / "basemlp_repeated_cv_submission_oof.csv"
BASELINE_REPEATED_FOLDS_PATH = REPORT_DIR / "basemlp_validation_fold_results.csv"
SPLIT_SEEDS = (42, 2026, 3407)
PUBLIC_SCORES = {
    "baseline_hardcoded": 0.12579,
    "raw_year_only": 0.12605,
    "blend_total_sf": 0.12564,
    "blend_total_bathrooms": 0.12538,
    "blend_quality_living_area": 0.12496,
}
PUBLIC_SCORE_SOURCE = (
    "baseline: prior user-confirmed Kaggle result; raw/blends: "
    "user-provided Kaggle screenshot dated 2026-07-20 KST"
)

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
assert train.shape == (1460, 81)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("external project script imports: 0")

train: 1,460 rows × 81 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
external project script imports: 0


## Feature construction

feature-engineering 폴더의 실제 식을 그대로 재현한다. 모든 feature는 target을
사용하지 않는 행 단위 수치이며 원본 열은 수정하지 않는다.

In [2]:
def clean_rows_without_record_correction(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned


def add_feature(cleaned: pd.DataFrame, feature_name: str) -> pd.DataFrame:
    featured = cleaned.copy()
    if feature_name == "TotalSF":
        components = ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]
        featured[feature_name] = featured[components].fillna(0).sum(axis=1)
    elif feature_name == "TotalBathrooms":
        featured[feature_name] = (
            featured["FullBath"].fillna(0)
            + 0.5 * featured["HalfBath"].fillna(0)
            + featured["BsmtFullBath"].fillna(0)
            + 0.5 * featured["BsmtHalfBath"].fillna(0)
        )
    elif feature_name == "QualityLivingArea":
        featured[feature_name] = (
            featured["OverallQual"] * featured["GrLivArea"]
        )
    else:
        raise ValueError(f"Unsupported feature: {feature_name}")
    return featured


raw_year_X = clean_rows_without_record_correction(
    train.drop(columns="SalePrice"), categorical_columns
)
assert float(
    raw_year_X.loc[raw_year_X["Id"].eq(524), "YearRemodAdd"].iloc[0]
) == 2008

MODEL_INPUTS = {
    "raw_year_only": raw_year_X,
    "total_sf_model": add_feature(raw_year_X, "TotalSF"),
    "total_bathrooms_model": add_feature(raw_year_X, "TotalBathrooms"),
    "quality_living_area_model": add_feature(
        raw_year_X, "QualityLivingArea"
    ),
}
MODEL_NUMERIC_COLUMNS = {
    "raw_year_only": list(NUMERIC_COLUMNS),
    "total_sf_model": [*NUMERIC_COLUMNS, "TotalSF"],
    "total_bathrooms_model": [*NUMERIC_COLUMNS, "TotalBathrooms"],
    "quality_living_area_model": [
        *NUMERIC_COLUMNS, "QualityLivingArea"
    ],
}

for model_name, frame in MODEL_INPUTS.items():
    generated = [
        column for column in frame.columns
        if column not in raw_year_X.columns
    ]
    assert len(generated) == (0 if model_name == "raw_year_only" else 1)
    assert raw_year_X.equals(frame.drop(columns=generated))
    if generated:
        assert frame[generated[0]].notna().all()
        assert np.isfinite(frame[generated[0]]).all()
        assert frame[generated[0]].ge(0).all()

display(pd.DataFrame([
    {
        "model": model_name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "generated_feature": next(
            (
                column for column in frame.columns
                if column not in raw_year_X.columns
            ),
            "None",
        ),
        "Id=524 YearRemodAdd": float(
            frame.loc[frame["Id"].eq(524), "YearRemodAdd"].iloc[0]
        ),
    }
    for model_name, frame in MODEL_INPUTS.items()
]))

def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )




,model,rows,columns,generated_feature,Id=524 YearRemodAdd
0,raw_year_only,1460,80,None,2008.0
1,total_sf_model,1460,81,TotalSF,2008.0
2,total_bathrooms_model,1460,81,TotalBathrooms,2008.0
3,quality_living_area_model,1460,81,QualityLivingArea,2008.0


## Repeated model fitting

raw-year reference와 세 one-feature support model을 각각 15 folds에서 학습한다.
총 60개 best checkpoint를 저장하며 seed 42 결과를 원본 feature 노트북과 대조한다.

In [3]:
OUTPUT_DIR = ARTIFACT_DIR / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fold_assignment_by_seed = np.zeros(
    (len(SPLIT_SEEDS), len(train)), dtype=np.int64
)
model_oof_by_variant: dict[str, np.ndarray] = {}
model_fold_rows: list[dict[str, object]] = []
started = time.perf_counter()

for model_index, (model_name, model_frame) in enumerate(
    MODEL_INPUTS.items(), start=1
):
    model_output_dir = OUTPUT_DIR / model_name
    checkpoint_dir = model_output_dir / "checkpoints"
    preprocessor_dir = model_output_dir / "preprocessors"
    epoch_log_dir = model_output_dir / "logs"
    for directory in (
        checkpoint_dir, preprocessor_dir, epoch_log_dir
    ):
        directory.mkdir(parents=True, exist_ok=True)

    oof_by_seed = np.full(
        (len(SPLIT_SEEDS), len(train)), np.nan, dtype=np.float64
    )
    for seed_index, split_seed in enumerate(SPLIT_SEEDS):
        splitter = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=split_seed,
        )
        for fold, (train_index, valid_index) in enumerate(
            splitter.split(model_frame), start=1
        ):
            split_number = seed_index * N_SPLITS + fold
            fold_seed = split_seed + fold
            fold_assignment_by_seed[seed_index, valid_index] = fold
            checkpoint_path = (
                checkpoint_dir /
                f"seed_{split_seed}_fold_{fold}_best.pt"
            )
            preprocessor_path = (
                preprocessor_dir /
                f"seed_{split_seed}_fold_{fold}_preprocessor.joblib"
            )
            epoch_log_path = (
                epoch_log_dir /
                f"seed_{split_seed}_fold_{fold}_epochs.csv"
            )

            preprocessor = make_preprocessor(
                MODEL_NUMERIC_COLUMNS[model_name]
            )
            X_train = preprocessor.fit_transform(
                model_frame.iloc[train_index]
            ).astype(np.float32)
            X_valid = preprocessor.transform(
                model_frame.iloc[valid_index]
            ).astype(np.float32)
            joblib.dump(preprocessor, preprocessor_path)

            model, training_result = train_fold(
                X_train,
                y_log[train_index],
                X_valid,
                y_log[valid_index],
                seed=fold_seed,
                experiment_id=f"{RUN_ID}-{model_name}",
                fold=split_number,
                checkpoint_path=checkpoint_path,
                epoch_log_path=epoch_log_path,
            )
            valid_log_prediction = predict_log_target(
                model,
                X_valid,
                training_result["target_mean"],
                training_result["target_std"],
            )
            oof_by_seed[seed_index, valid_index] = valid_log_prediction
            model_fold_rows.append({
                "model": model_name,
                "split_seed": split_seed,
                "fold": fold,
                "model_seed": fold_seed,
                "training_rows": len(train_index),
                "validation_rows": len(valid_index),
                "encoded_features": X_train.shape[1],
                "best_epoch": training_result["best_epoch"],
                "stopping_epoch": training_result["stopping_epoch"],
                "rmsle": training_result[
                    "restored_validation_rmsle"
                ],
                "checkpoint_path": str(
                    checkpoint_path.relative_to(ROOT)
                ),
                "preprocessor_path": str(
                    preprocessor_path.relative_to(ROOT)
                ),
                "epoch_log_path": str(
                    epoch_log_path.relative_to(ROOT)
                ),
            })
    assert not np.isnan(oof_by_seed).any()
    model_oof_by_variant[model_name] = oof_by_seed
    current_scores = [
        row["rmsle"] for row in model_fold_rows
        if row["model"] == model_name
    ]
    print(
        f"[{model_index}/4] {model_name}: "
        f"15-fold mean={np.mean(current_scores):.6f}"
    )

model_fold_results = pd.DataFrame(model_fold_rows)
model_fold_results.to_csv(MODEL_FOLD_RESULTS_PATH, index=False)
assert len(model_fold_results) == 60
assert not model_fold_results.duplicated(
    ["model", "split_seed", "fold"]
).any()

# Seed-42 must exactly reproduce the four original feature-engineering notebooks.
expected_metric_paths = {
    "raw_year_only": (
        ROOT / "reports" /
        "basemlp_metrics_BASEMLP-20260719-RAWYEAR-ONLY-NB-04.json"
    ),
    "total_sf_model": (
        ROOT / "reports" /
        "basemlp_metrics_BASEMLP-20260719-FEAT-TOTALSF-NB-06.json"
    ),
    "total_bathrooms_model": (
        ROOT / "reports" /
        "basemlp_metrics_BASEMLP-20260719-FEAT-TOTALBATH-NB-07.json"
    ),
    "quality_living_area_model": (
        ROOT / "reports" /
        "basemlp_metrics_BASEMLP-20260719-FEAT-QUALAREA-NB-08.json"
    ),
}
for model_name, metrics_path in expected_metric_paths.items():
    expected = json.loads(
        metrics_path.read_text(encoding="utf-8")
    )["results"]["fold_scores"]
    observed = model_fold_results.loc[
        model_fold_results["model"].eq(model_name)
        & model_fold_results["split_seed"].eq(42)
    ].sort_values("fold")["rmsle"].to_numpy()
    assert np.allclose(expected, observed, rtol=0, atol=1e-12)

duration_seconds = time.perf_counter() - started
print(f"60 model fits completed in {duration_seconds:.2f} seconds")
print("All seed-42 model folds exactly reproduce the source notebooks.")

[1/4] raw_year_only: 15-fold mean=0.145635


[2/4] total_sf_model: 15-fold mean=0.145674


[3/4] total_bathrooms_model: 15-fold mean=0.147008


[4/4] quality_living_area_model: 15-fold mean=0.145410
60 model fits completed in 20.39 seconds
All seed-42 model folds exactly reproduce the source notebooks.


## Cross-fitted blends

각 seed 안에서 현재 fold를 제외한 OOF 행으로만 scalar feature weight를 계산한다.
seed 42의 fold 점수와 weight가 기존 blend 노트북을 정확히 재현하는지도 확인한다.

In [4]:
def fit_convex_weight(
    actual_log: np.ndarray,
    raw_log: np.ndarray,
    feature_log: np.ndarray,
) -> float:
    direction = feature_log - raw_log
    denominator = float(np.sum(direction ** 2))
    if denominator == 0:
        return 0.0
    numerator = -float(
        np.sum((raw_log - actual_log) * direction)
    )
    return float(np.clip(numerator / denominator, 0.0, 1.0))


baseline_oof = pd.read_csv(BASELINE_REPEATED_OOF_PATH)
assert baseline_oof["Id"].equals(train["Id"])
baseline_prediction_by_seed = np.vstack([
    baseline_oof[
        f"seed_{split_seed}_oof_log_prediction"
    ].to_numpy(dtype=np.float64)
    for split_seed in SPLIT_SEEDS
])

candidate_prediction_by_seed = {
    "baseline_hardcoded": baseline_prediction_by_seed,
    "raw_year_only": model_oof_by_variant["raw_year_only"],
}
candidate_fold_rows: list[dict[str, object]] = []

baseline_fold_source = pd.read_csv(BASELINE_REPEATED_FOLDS_PATH)
baseline_fold_source = baseline_fold_source.loc[
    baseline_fold_source["base_strategy"].eq(
        "repeated_kfold5_3seeds"
    )
    & baseline_fold_source["policy"].eq("keep_all")
]
assert len(baseline_fold_source) == 15
for row in baseline_fold_source.to_dict(orient="records"):
    candidate_fold_rows.append({
        "candidate": "baseline_hardcoded",
        "split_seed": int(row["component_seed"]),
        "fold": int(row["fold"]),
        "feature_weight": np.nan,
        "raw_weight": np.nan,
        "rmsle": float(row["rmsle"]),
    })

raw_model_folds = model_fold_results.loc[
    model_fold_results["model"].eq("raw_year_only")
]
for row in raw_model_folds.to_dict(orient="records"):
    candidate_fold_rows.append({
        "candidate": "raw_year_only",
        "split_seed": int(row["split_seed"]),
        "fold": int(row["fold"]),
        "feature_weight": np.nan,
        "raw_weight": np.nan,
        "rmsle": float(row["rmsle"]),
    })

blend_specs = {
    "blend_total_sf": "total_sf_model",
    "blend_total_bathrooms": "total_bathrooms_model",
    "blend_quality_living_area": "quality_living_area_model",
}
final_weights_by_candidate: dict[str, list[float]] = {}

for candidate, feature_model in blend_specs.items():
    raw_predictions = model_oof_by_variant["raw_year_only"]
    feature_predictions = model_oof_by_variant[feature_model]
    blend_predictions = np.full_like(raw_predictions, np.nan)
    final_weights = []
    for seed_index, split_seed in enumerate(SPLIT_SEEDS):
        fold_assignment = fold_assignment_by_seed[seed_index]
        final_weights.append(fit_convex_weight(
            y_log,
            raw_predictions[seed_index],
            feature_predictions[seed_index],
        ))
        for fold in range(1, N_SPLITS + 1):
            meta_train_mask = fold_assignment != fold
            meta_valid_mask = fold_assignment == fold
            feature_weight = fit_convex_weight(
                y_log[meta_train_mask],
                raw_predictions[seed_index, meta_train_mask],
                feature_predictions[seed_index, meta_train_mask],
            )
            prediction = (
                (1.0 - feature_weight)
                * raw_predictions[seed_index, meta_valid_mask]
                + feature_weight
                * feature_predictions[seed_index, meta_valid_mask]
            )
            blend_predictions[seed_index, meta_valid_mask] = prediction
            candidate_fold_rows.append({
                "candidate": candidate,
                "split_seed": split_seed,
                "fold": fold,
                "feature_weight": feature_weight,
                "raw_weight": 1.0 - feature_weight,
                "rmsle": float(np.sqrt(np.mean(
                    (y_log[meta_valid_mask] - prediction) ** 2
                ))),
            })
    assert not np.isnan(blend_predictions).any()
    candidate_prediction_by_seed[candidate] = blend_predictions
    final_weights_by_candidate[candidate] = final_weights

candidate_fold_results = pd.DataFrame(candidate_fold_rows)
candidate_fold_results.to_csv(CANDIDATE_FOLD_RESULTS_PATH, index=False)
assert len(candidate_fold_results) == 75
assert not candidate_fold_results.duplicated(
    ["candidate", "split_seed", "fold"]
).any()

# The seed-42 blends must reproduce the original cross-fitted notebooks.
expected_blend_paths = {
    "blend_total_sf": ROOT / "reports/basemlp_blend_total_sf_comparison.json",
    "blend_total_bathrooms": ROOT / "reports/basemlp_blend_total_bathrooms_comparison.json",
    "blend_quality_living_area": ROOT / "reports/basemlp_blend_quality_living_interaction_comparison.json",
}
for candidate, comparison_path in expected_blend_paths.items():
    expected = json.loads(comparison_path.read_text(encoding="utf-8"))
    observed = candidate_fold_results.loc[
        candidate_fold_results["candidate"].eq(candidate)
        & candidate_fold_results["split_seed"].eq(42)
    ].sort_values("fold")
    assert np.allclose(
        expected["fold_scores"], observed["rmsle"].to_numpy(),
        rtol=0, atol=1e-12,
    )
    assert np.allclose(
        expected["crossfit_weights"],
        observed["feature_weight"].to_numpy(),
        rtol=0, atol=1e-12,
    )

print("All seed-42 blend scores and weights exactly reproduce the source notebooks.")
display(
    candidate_fold_results.groupby("candidate")["rmsle"]
    .agg(["mean", "std", "count"]).round(6)
)

All seed-42 blend scores and weights exactly reproduce the source notebooks.


,mean,std,count
candidate,,,
baseline_hardcoded,0.145881,0.026967,15
blend_quality_living_area,0.136490,0.026533,15
blend_total_bathrooms,0.136764,0.027616,15
blend_total_sf,0.136563,0.026760,15
raw_year_only,0.145635,0.026512,15


## Results

15-fold CV, 세 seed의 전체 held-out 예측을 합친 repeated OOF RMSLE,
행별 세 OOF 예측을 평균한 3-seed ensemble OOF RMSLE를 분리해 보고한다.
네 변경의 reference 대비 방향과 전체 후보 순위를 Kaggle public과 비교한다.

In [5]:
oof_frames = []
for seed_index, split_seed in enumerate(SPLIT_SEEDS):
    frame = pd.DataFrame({
        "Id": train["Id"].to_numpy(dtype=np.int64),
        "split_seed": split_seed,
        "fold": fold_assignment_by_seed[seed_index],
        "actual_log1p": y_log,
    })
    for candidate, predictions in candidate_prediction_by_seed.items():
        frame[f"{candidate}_log_prediction"] = predictions[seed_index]
    for model_name, predictions in model_oof_by_variant.items():
        if model_name != "raw_year_only":
            frame[f"support_{model_name}_log_prediction"] = (
                predictions[seed_index]
            )
    oof_frames.append(frame)
oof_results = pd.concat(oof_frames, ignore_index=True)
oof_results.to_csv(OOF_RESULTS_PATH, index=False)
assert len(oof_results) == len(train) * len(SPLIT_SEEDS)
assert np.isfinite(
    oof_results.select_dtypes(include="number").to_numpy()
).all()

candidate_order = [
    "baseline_hardcoded",
    "raw_year_only",
    "blend_total_sf",
    "blend_total_bathrooms",
    "blend_quality_living_area",
]
result_rows = []
for candidate in candidate_order:
    predictions = candidate_prediction_by_seed[candidate]
    fold_scores = candidate_fold_results.loc[
        candidate_fold_results["candidate"].eq(candidate), "rmsle"
    ].to_numpy(dtype=np.float64)
    ensemble_prediction = predictions.mean(axis=0)
    result_rows.append({
        "candidate": candidate,
        "cv_mean": float(fold_scores.mean()),
        "cv_std": float(fold_scores.std(ddof=1)),
        "repeated_oof_rmsle": float(np.sqrt(np.mean(
            (np.tile(y_log, (len(SPLIT_SEEDS), 1)) - predictions) ** 2
        ))),
        "three_seed_ensemble_oof_rmsle": float(np.sqrt(np.mean(
            (y_log - ensemble_prediction) ** 2
        ))),
        "public_score": PUBLIC_SCORES[candidate],
        "improved_folds_vs_raw": (
            np.nan if candidate in {
                "baseline_hardcoded", "raw_year_only"
            } else int(np.sum(
                fold_scores
                < candidate_fold_results.loc[
                    candidate_fold_results["candidate"].eq(
                        "raw_year_only"
                    ), "rmsle"
                ].to_numpy(dtype=np.float64)
            ))
        ),
        "mean_crossfit_feature_weight": (
            np.nan if candidate not in blend_specs
            else float(candidate_fold_results.loc[
                candidate_fold_results["candidate"].eq(candidate),
                "feature_weight",
            ].mean())
        ),
        "std_crossfit_feature_weight": (
            np.nan if candidate not in blend_specs
            else float(candidate_fold_results.loc[
                candidate_fold_results["candidate"].eq(candidate),
                "feature_weight",
            ].std(ddof=1))
        ),
    })

results = pd.DataFrame(result_rows)
for metric in [
    "cv_mean", "repeated_oof_rmsle",
    "three_seed_ensemble_oof_rmsle", "public_score",
]:
    results[f"{metric}_rank"] = results[metric].rank(
        method="min", ascending=True
    ).astype(int)
results.to_csv(RESULTS_PATH, index=False)

reference_map = {
    "raw_year_only": "baseline_hardcoded",
    "blend_total_sf": "raw_year_only",
    "blend_total_bathrooms": "raw_year_only",
    "blend_quality_living_area": "raw_year_only",
}
hypothesis_rows = []
for candidate, reference_candidate in reference_map.items():
    candidate_row = results.loc[
        results["candidate"].eq(candidate)
    ].iloc[0]
    reference_row = results.loc[
        results["candidate"].eq(reference_candidate)
    ].iloc[0]
    cv_delta = float(candidate_row["cv_mean"] - reference_row["cv_mean"])
    oof_delta = float(
        candidate_row["repeated_oof_rmsle"]
        - reference_row["repeated_oof_rmsle"]
    )
    ensemble_delta = float(
        candidate_row["three_seed_ensemble_oof_rmsle"]
        - reference_row["three_seed_ensemble_oof_rmsle"]
    )
    public_delta = float(
        candidate_row["public_score"] - reference_row["public_score"]
    )
    hypothesis_rows.append({
        "experiment": candidate,
        "reference": reference_candidate,
        "cv_delta": cv_delta,
        "repeated_oof_delta": oof_delta,
        "three_seed_ensemble_oof_delta": ensemble_delta,
        "public_delta": public_delta,
        "cv_direction_matches_public": bool(
            np.sign(cv_delta) == np.sign(public_delta)
        ),
        "oof_direction_matches_public": bool(
            np.sign(oof_delta) == np.sign(public_delta)
        ),
        "ensemble_direction_matches_public": bool(
            np.sign(ensemble_delta) == np.sign(public_delta)
        ),
    })
hypothesis_results = pd.DataFrame(hypothesis_rows)
hypothesis_results.to_csv(HYPOTHESIS_RESULTS_PATH, index=False)

def pairwise_rank_agreement(metric: str) -> dict[str, object]:
    agreements = 0
    comparisons = 0
    for left in range(len(results)):
        for right in range(left + 1, len(results)):
            local_difference = (
                results.iloc[left][metric] - results.iloc[right][metric]
            )
            public_difference = (
                results.iloc[left]["public_score"]
                - results.iloc[right]["public_score"]
            )
            agreements += int(
                np.sign(local_difference) == np.sign(public_difference)
            )
            comparisons += 1
    return {
        "agreements": agreements,
        "comparisons": comparisons,
        "rate": agreements / comparisons,
    }

rank_metrics = [
    "cv_mean", "repeated_oof_rmsle",
    "three_seed_ensemble_oof_rmsle",
]
rank_alignment = {
    metric: {
        "spearman_with_public": float(
            results[metric].corr(results["public_score"], method="spearman")
        ),
        "kendall_with_public": float(
            results[metric].corr(results["public_score"], method="kendall")
        ),
        "pairwise": pairwise_rank_agreement(metric),
        "order": results.sort_values(metric)["candidate"].tolist(),
        "exact_public_order": bool(
            results.sort_values(metric)["candidate"].tolist()
            == results.sort_values("public_score")["candidate"].tolist()
        ),
    }
    for metric in rank_metrics
}
rank_alignment["public_order"] = results.sort_values(
    "public_score"
)["candidate"].tolist()

summary = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_group": RUN_ID,
    "question": (
        "Do repeated-CV CV/OOF movements reproduce the direction and ranking "
        "of four already-submitted BaseMLP changes?"
    ),
    "public_score_source": PUBLIC_SCORE_SOURCE,
    "public_scores_used_after_training_only": True,
    "validation": {
        "splitter": "KFold(5, shuffle=True)",
        "split_seeds": list(SPLIT_SEEDS),
        "model_fits": int(len(model_fold_results)),
        "candidate_fold_rows": int(len(candidate_fold_results)),
        "raw_year_id524_yearremodadd": 2008,
        "external_project_script_imports": 0,
        "seed42_source_model_scores_exactly_reproduced": True,
        "seed42_source_blend_scores_and_weights_exactly_reproduced": True,
    },
    "rank_alignment": rank_alignment,
    "hypothesis_direction_match_counts": {
        "cv": int(hypothesis_results[
            "cv_direction_matches_public"
        ].sum()),
        "repeated_oof": int(hypothesis_results[
            "oof_direction_matches_public"
        ].sum()),
        "three_seed_ensemble_oof": int(hypothesis_results[
            "ensemble_direction_matches_public"
        ].sum()),
        "total": int(len(hypothesis_results)),
    },
    "duration_seconds": duration_seconds,
    "artifacts": {
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "model_fold_results": str(MODEL_FOLD_RESULTS_PATH.relative_to(ROOT)),
        "candidate_fold_results": str(CANDIDATE_FOLD_RESULTS_PATH.relative_to(ROOT)),
        "oof_predictions": str(OOF_RESULTS_PATH.relative_to(ROOT)),
        "results": str(RESULTS_PATH.relative_to(ROOT)),
        "hypotheses": str(HYPOTHESIS_RESULTS_PATH.relative_to(ROOT)),
        "artifact_dir": str(OUTPUT_DIR.relative_to(ROOT)),
    },
    "private_scores": "unverified",
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

run_timestamp = summary["run_timestamp_utc"]
experiment_ids = {
    "raw_year_only": f"{RUN_ID}-RAWYEAR",
    "blend_total_sf": f"{RUN_ID}-TOTALSF-BLEND",
    "blend_total_bathrooms": f"{RUN_ID}-TOTALBATH-BLEND",
    "blend_quality_living_area": f"{RUN_ID}-QUALAREA-BLEND",
}
for hypothesis in hypothesis_results.to_dict(orient="records"):
    candidate = hypothesis["experiment"]
    candidate_row = results.loc[
        results["candidate"].eq(candidate)
    ].iloc[0]
    if candidate == "raw_year_only":
        model_description = "Manual PyTorch BaseMLP"
        feature_description = "79 original predictors; raw YearRemodAdd; Id excluded"
        architecture = "input->128(ReLU)->64(ReLU)->1"
        checkpoint_path = str(
            (OUTPUT_DIR / "raw_year_only/checkpoints").relative_to(ROOT)
        )
    else:
        support_model = blend_specs[candidate]
        model_description = "Cross-fitted log-space convex blend of two manual BaseMLPs"
        feature_description = (
            f"raw-year BaseMLP + {support_model} BaseMLP prediction blend"
        )
        architecture = "fold-external closed-form convex weight; raw + one-feature BaseMLP"
        checkpoint_path = " | ".join([
            str((OUTPUT_DIR / "raw_year_only/checkpoints").relative_to(ROOT)),
            str((OUTPUT_DIR / support_model / "checkpoints").relative_to(ROOT)),
        ])
    append_experiment({
        "experiment_id": experiment_ids[candidate],
        "datetime": run_timestamp,
        "objective": (
            "Check whether repeated-CV CV/OOF direction matches the already-observed Kaggle public movement"
        ),
        "data_version": f"train sha256={sha256(TRAIN_PATH)}",
        "split_strategy": "KFold(5,shuffle=True) repeated at seeds 42|2026|3407; blend weights fitted on other OOF folds within each seed",
        "folds": "5x3",
        "seed": "42|2026|3407; model seed=split_seed+fold",
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": "Raw YearRemodAdd; fold median/scaling/one-hot; fold-train target standardization; all rows retained",
        "features": feature_description,
        "model": model_description,
        "architecture": architecture,
        "optimizer": "Adam for BaseMLP components",
        "loss": "MSELoss on fold-standardized log target; closed-form log-MSE blend weight where applicable",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "hyperparameters": json.dumps({
            "candidate": candidate,
            "reference": hypothesis["reference"],
            "split_seeds": list(SPLIT_SEEDS),
            "blend_space": "log1p(SalePrice)",
            "public_score_used_for_training": False,
            "source_notebooks_exactly_reproduced_at_seed42": True,
        }, ensure_ascii=False),
        "cv_mean": candidate_row["cv_mean"],
        "cv_std": candidate_row["cv_std"],
        "holdout_score": candidate_row[
            "three_seed_ensemble_oof_rmsle"
        ],
        "kaggle_public_score": candidate_row["public_score"],
        "checkpoint_path": checkpoint_path,
        "artifact_path": " | ".join([
            str(NOTEBOOK_PATH.relative_to(ROOT)),
            str(RESULTS_PATH.relative_to(ROOT)),
            str(HYPOTHESIS_RESULTS_PATH.relative_to(ROOT)),
            str(OOF_RESULTS_PATH.relative_to(ROOT)),
            str(SUMMARY_PATH.relative_to(ROOT)),
        ]),
        "result": (
            "direction_match" if hypothesis[
                "cv_direction_matches_public"
            ] else "direction_mismatch"
        ),
        "interpretation": (
            f"vs {hypothesis['reference']}: "
            f"CV delta={hypothesis['cv_delta']:+.6f}; "
            f"repeated OOF delta={hypothesis['repeated_oof_delta']:+.6f}; "
            f"3-seed ensemble OOF delta={hypothesis['three_seed_ensemble_oof_delta']:+.6f}; "
            f"public delta={hypothesis['public_delta']:+.5f}."
        ),
        "next_action": "Use this as validation-alignment evidence; keep public leaderboard out of further feature selection",
    })

display(results.round(6))
display(hypothesis_results.round(6))
display(pd.DataFrame([
    {
        "metric": metric,
        "spearman_with_public": rank_alignment[metric]["spearman_with_public"],
        "kendall_with_public": rank_alignment[metric]["kendall_with_public"],
        "pairwise_agreement_rate": rank_alignment[metric]["pairwise"]["rate"],
        "exact_public_order": rank_alignment[metric]["exact_public_order"],
    }
    for metric in rank_metrics
]))
print(f"Summary: {SUMMARY_PATH.relative_to(ROOT)}")

,candidate,cv_mean,cv_std,repeated_oof_rmsle,three_seed_ensemble_oof_rmsle,public_score,improved_folds_vs_raw,mean_crossfit_feature_weight,std_crossfit_feature_weight,cv_mean_rank,repeated_oof_rmsle_rank,three_seed_ensemble_oof_rmsle_rank,public_score_rank
0,baseline_hardcoded,0.145881,0.026967,0.148189,0.134232,0.12579,NaN,NaN,NaN,5,5,5,4
1,raw_year_only,0.145635,0.026512,0.147870,0.133851,0.12605,NaN,NaN,NaN,4,4,4,5
2,blend_total_sf,0.136563,0.026760,0.138988,0.130240,0.12564,15.0,0.497863,0.108723,2,2,2,3
3,blend_total_bathrooms,0.136764,0.027616,0.139342,0.130942,0.12538,14.0,0.475051,0.117096,3,3,3,2
4,blend_quality_living_area,0.136490,0.026533,0.138876,0.130110,0.12496,15.0,0.501891,0.094743,1,1,1,1


,experiment,reference,cv_delta,repeated_oof_delta,three_seed_ensemble_oof_delta,public_delta,cv_direction_matches_public,oof_direction_matches_public,ensemble_direction_matches_public
0,raw_year_only,baseline_hardcoded,-0.000246,-0.000319,-0.000381,0.00026,False,False,False
1,blend_total_sf,raw_year_only,-0.009072,-0.008882,-0.003611,-0.00041,True,True,True
2,blend_total_bathrooms,raw_year_only,-0.008871,-0.008528,-0.002910,-0.00067,True,True,True
3,blend_quality_living_area,raw_year_only,-0.009145,-0.008994,-0.003741,-0.00109,True,True,True


,metric,spearman_with_public,kendall_with_public,pairwise_agreement_rate,exact_public_order
0,cv_mean,0.8,0.6,0.8,False
1,repeated_oof_rmsle,0.8,0.6,0.8,False
2,three_seed_ensemble_oof_rmsle,0.8,0.6,0.8,False


Summary: reports/validation_test/basemlp_feature_rank_summary.json


## Takeaways

- 세 feature blend는 raw-year 대비 반복 CV에서 0.008871~0.009145, 반복 OOF에서 0.008528~0.008994 개선됐고 public에서도 0.00041~0.00109 개선됐다. 방향은 3/3 일치하지만 local 개선 크기는 public에 비해 크게 과장되어 절대값 보정에는 사용할 수 없다.
- QualityLivingArea blend는 CV 0.136490, 반복 OOF 0.138876, 3-seed ensemble OOF 0.130110으로 세 local 관점과 public 0.12496에서 모두 1위다.
- TotalSF는 local 2위, TotalBathrooms는 public 2위라 두 후보의 근소한 순서는 불일치한다.
- hardcoded baseline과 raw-year의 local CV 차이는 0.000246으로 fold SD 약 0.027보다 매우 작다. 이 수준의 차이를 validation 선택 신호로 쓰면 안 된다.
- 전체 순위 상관 Spearman 0.80과 pairwise 8/10은 방향성 검증에는 긍정적이지만 후보가 5개뿐이고 public 결과를 사후 비교했으므로 Share with caveats로 해석한다.
- public 점수는 학습이나 blend weight 추정에 사용하지 않았으며 private 점수는 모두 unverified다.